In [1]:
# 1. Install libraries (run once)
%pip install pandas scikit-learn matplotlib nltk

# 2. Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import re
import nltk

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("italian") + stopwords.words("english"))

# 3. Load dataset
df = pd.read_csv("firenze.csv", encoding="utf-8-sig")
print(df.head())
print(df.columns)

# 4. Select text columns automatically
cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
if not cols:
    raise ValueError("No text columns found in the CSV.")

df["text"] = df[cols].fillna("").astype(str).agg(" ".join, axis=1)

# 5. Clean text
def clean_text(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text.lower())
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(
        w for w in text.split()
        if w not in stop_words and len(w) > 1
    )

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.strip() != ""].copy()

if len(df) < 3:
    raise ValueError("Need at least 3 rows containing text.")

# 6. NLP: Convert text into TF-IDF features
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X = tfidf.fit_transform(df["clean_text"])

# 7. Find optimal K using the elbow method
ks = range(2, min(10, len(df)))
inertia = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertia.append(model.inertia_)

plt.plot(list(ks), inertia, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# 8. Apply K-Means (change K if needed)
K = min(5, len(df) - 1)
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 9. Evaluate clustering
if len(set(df["cluster"])) > 1:
    print("Silhouette Score:", round(silhouette_score(X, df["cluster"]), 3))

# 10. Display keywords per cluster
terms = tfidf.get_feature_names_out()
for i, center in enumerate(kmeans.cluster_centers_):
    print(f"Cluster {i}:", ", ".join(terms[center.argsort()[-10:][::-1]]))

# 11. Visualize clusters
if X.shape[1] >= 2:
    coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(X)
    plt.scatter(coords[:, 0], coords[:, 1], c=df["cluster"], cmap="viridis")
    plt.title("Firenze NLP K-Means Clusters")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

# 12. Save results
df.to_csv("firenze_kmeans_results.csv", index=False, encoding="utf-8-sig")
print("Saved: firenze_kmeans_results.csv")
print(df[["clean_text", "cluster"]].head())

Note: you may need to restart the kernel to use updated packages.
            Acciaiuoli  Albizzi  Barbadori  Bischeri  Castellani  Ginori  \
Acciaiuoli           0        0          0         0           0       0   
Albizzi              0        0          0         0           0       1   
Barbadori            0        0          0         0           1       0   
Bischeri             0        0          0         0           0       0   
Castellani           0        0          1         0           0       0   

            Guadagni  Lamberteschi  Medici  Pazzi  Peruzzi  Pucci  Ridolfi  \
Acciaiuoli         0             0       1      0        0      0        0   
Albizzi            1             0       1      0        0      0        0   
Barbadori          0             0       1      0        0      0        0   
Bischeri           1             0       0      0        1      0        0   
Castellani         0             0       0      0        1      0        0   

        

ValueError: No text columns found in the CSV.